# Section 7 · Observability 3 — Langfuse (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

[Langfuse](https://langfuse.com) is an open-source LLM observability platform: cost tracking, prompt versioning, scoring and nested trace views. Use the managed cloud or self-host it.

**Get your keys**
1. Sign up at https://cloud.langfuse.com (EU) or https://us.cloud.langfuse.com (US)
2. Create a project → **Settings → API Keys → Create new API keys**
3. Copy the **Public Key** (`pk-lf-...`) and **Secret Key** (`sk-lf-...`)

The Langfuse SDK registers itself as the OpenTelemetry tracer provider, so once the client is initialised, Strands spans flow to Langfuse automatically.

> Each tracing notebook should run in a **fresh kernel**: OpenTelemetry allows only one global tracer provider per Python process.

In [1]:
%pip install -q -r requirements-groq.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv(override=True)  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")
API_KEY = os.getenv("GROQ_API_KEY")
BASE_URL = os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1")  # or a local OpenAI-compatible server

model = OpenAIModel(
    client_args={
        "api_key": API_KEY,
        "base_url": BASE_URL,  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    # params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")
print(f"Using {BASE_URL} as the base URL")

Using openai/gpt-oss-120b on Groq
Using https://api.groq.com/openai/v1 as the base URL


In [3]:
# Your Langfuse project keys (or export them in your shell before starting Jupyter)
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_BASE_URL = os.getenv("LANGFUSE_BASE_URL")

print(f"Using Langfuse public key: {LANGFUSE_PUBLIC_KEY}")
print(f"Using Langfuse secret key: {LANGFUSE_SECRET_KEY}")
print(f"Using Langfuse base URL: {LANGFUSE_BASE_URL}")

Using Langfuse public key: pk-lf-506492f8-d9b6-4c4c-88ed-a6ed66a6f6a1
Using Langfuse secret key: sk-lf-d31d05e7-9b0d-4492-92be-89c088c9aba5
Using Langfuse base URL: https://us.cloud.langfuse.com


In [4]:
from langfuse import get_client

langfuse = get_client()

if langfuse.auth_check():
    print("Langfuse client is authenticated and ready.")
else:
    print("Authentication failed. Check your keys and base URL.")

Langfuse client is authenticated and ready.


In [5]:
from strands import Agent

agent = Agent(
    model=model,
    system_prompt="You are a document analyst. Summarize documents clearly and concisely.",
    trace_attributes={"session.id": "langfuse-demo-session", "user.id": "learner@example.com"},
)

response = agent("What are the key risks in a standard SaaS subscription agreement?")

# Ensure traces are flushed to Langfuse before the notebook moves on
langfuse.flush()

The user asks: "What are the key risks in a standard SaaS subscription agreement?" As a document analyst, need to summarize clearly and concisely. Provide list of key risks: service availability/SLAs, data security & privacy, termination & exit, IP ownership, indemnification, limitation of liability, change of terms, fees & payment, compliance, jurisdiction, force majeure. Provide brief explanation for each. No disallowed content. Should be concise.**Key Risks in a Standard SaaS Subscription Agreement**

| Risk | Why It Matters | Typical Mitigation |
|------|----------------|--------------------|
| **1. Service Availability / Performance** | Downtime or slow performance can disrupt business operations. | Look for clear Service Level Agreements (SLAs) with uptime percentages, response‑time metrics, and remedies (credits, termination rights). |
| **2. Data Security & Privacy** | Breaches can lead to regulatory fines, reputational damage, and loss of customer trust. | Require industry‑sta

Open your Langfuse project → **Tracing**. You'll see the system prompt and user message, each model call with tokens and latency, any tool calls, and an estimated cost.